# 0. Read files and Feature Engineering

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import warnings
import os

warnings.filterwarnings('ignore')

# Device selection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3090


In [2]:
# Load data
print("Loading data...")

input_path = 'trademaster25' if os.path.exists('trademaster25') else '/kaggle/input/trademaster'

train_df = pd.read_csv(f'{input_path}/train_v2.csv')
test_df = pd.read_csv(f'{input_path}/test_v2.csv')

print(f"Training set: {train_df.shape}")
print(f"Test set: {test_df.shape}")

# Feature and target columns
feature_cols = [f'feature_{i}' for i in range(1, 27)]+[f'feature_{j}' for j in range(28, 31)]
target_cols = ['target_short', 'target_medium', 'target_long']
time_cols = ['date_id', 'minute_id']
TARGET_WEIGHTS = {'short': 0.5, 'medium': 0.3, 'long': 0.2}

print(f"Features: {len(feature_cols)}")
print(f"Targets: {target_cols}")
print(f"Weights: {TARGET_WEIGHTS}")

Loading data...
Training set: (139392, 37)
Test set: (34348, 34)
Features: 29
Targets: ['target_short', 'target_medium', 'target_long']
Weights: {'short': 0.5, 'medium': 0.3, 'long': 0.2}


In [3]:
# Fill NaN with median and clip extreme values
for col in feature_cols:
    median_val = train_df[col].median()
    train_df[col].fillna(median_val, inplace=True)
    test_df[col].fillna(median_val, inplace=True)

    # Clip extreme values at 1% and 99% quantiles
    lower = train_df[col].quantile(0.01)
    upper = train_df[col].quantile(0.99)
    train_df[col] = train_df[col].clip(lower, upper)
    test_df[col] = test_df[col].clip(lower, upper)

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.preprocessing import QuantileTransformer

print("=" * 80)
print("识别连续特征和离散特征")
print("=" * 80)

# 识别离散特征（唯一值较少的特征）
discrete_features = []
continuous_features = []

for col in feature_cols:
    unique_count = train_df[col].nunique()
    unique_vals = train_df[col].unique()
    
    # 如果唯一值 <= 10 且都是整数，认为是离散特征
    if unique_count <= 10:
        is_integer = all(train_df[col].dropna().apply(lambda x: x == int(x)))
        if is_integer or unique_count <= 3:
            discrete_features.append(col)
            print(f"{col}: 离散特征 (唯一值={unique_count}, 值={sorted(unique_vals[:10])})")
        else:
            continuous_features.append(col)
    else:
        continuous_features.append(col)

print(f"\n发现 {len(discrete_features)} 个离散特征")
print(f"发现 {len(continuous_features)} 个连续特征")
print(f"\n离散特征列表: {discrete_features}")

print("\n" + "=" * 80)
print("GaussRank 归一化 (仅对连续特征)")
print("=" * 80)

if len(continuous_features) > 0:
    # 对连续特征进行 GaussRank 归一化
    gauss_scaler = QuantileTransformer(output_distribution='normal', random_state=42)
    
    print(f"对 {len(continuous_features)} 个连续特征应用 GaussRank 归一化...")
    train_df[continuous_features] = gauss_scaler.fit_transform(train_df[continuous_features])
    test_df[continuous_features] = gauss_scaler.transform(test_df[continuous_features])
    
    print("GaussRank 归一化完成。")
    
    # 验证归一化后的分布
    print("\n归一化后的偏度和峰度 (前5个连续特征):")
    print(train_df[continuous_features[:5]].agg(['skew', 'kurtosis']))

if len(discrete_features) > 0:
    print("\n" + "=" * 80)
    print("离散特征处理")
    print("=" * 80)
    print(f"离散特征保持原样，不进行归一化")
    print(f"这些特征的离散性质在树模型中会被保留")
    
    # 显示离散特征的分布
    print("\n离散特征分布:")
    for col in discrete_features[:5]:  # 只显示前5个
        value_counts = train_df[col].value_counts().sort_index()
        print(f"\n{col}:")
        print(value_counts)


识别连续特征和离散特征
feature_1: 离散特征 (唯一值=2, 值=[np.int64(0), np.int64(1)])

发现 1 个离散特征
发现 28 个连续特征

离散特征列表: ['feature_1']

GaussRank 归一化 (仅对连续特征)
对 28 个连续特征应用 GaussRank 归一化...
GaussRank 归一化完成。

归一化后的偏度和峰度 (前5个连续特征):
          feature_2  feature_3  feature_4  feature_5  feature_6
skew       0.010780  -0.031259   0.000651  -0.008955   0.002009
kurtosis   5.505653   5.551395   5.514407   5.330535   6.007597

离散特征处理
离散特征保持原样，不进行归一化
这些特征的离散性质在树模型中会被保留

离散特征分布:

feature_1:
feature_1
0    125362
1     14030
Name: count, dtype: int64


# 1. Advanced Training Framework with Cross-Validation

In [8]:
import lightgbm as lgb
import catboost as cbt
import joblib

# Training configuration
TRAINING = True  # Set to True for training, False for loading pre-trained models
N_fold = 5  # Number of folds for cross-validation
num_valid_dates = 117  # Number of dates for validation
skip_dates = 0  # Number of dates to skip from the beginning

# Create directory for saving models
os.makedirs('models', exist_ok=True)

print(f"Configuration:")
print(f"  Training Mode: {TRAINING}")
print(f"  Number of Folds: {N_fold}")
print(f"  Validation Dates: {num_valid_dates}")
print(f"  Skip Dates: {skip_dates}")

Configuration:
  Training Mode: True
  Number of Folds: 5
  Validation Dates: 117
  Skip Dates: 0


In [ ]:
# Prepare time-based cross-validation splits
if TRAINING:
    # Filter data after skip_dates
    train_df_filtered = train_df[train_df['date_id'] >= skip_dates].reset_index(drop=True)
    
    # Get unique dates
    dates = train_df_filtered['date_id'].unique()
    print(f"Total unique dates: {len(dates)}")
    
    # Define validation and training dates
    valid_dates = dates[-num_valid_dates:]
    train_dates = dates[:-num_valid_dates]
    
    print(f"Training dates: {len(train_dates)}")
    print(f"Validation dates: {len(valid_dates)}")
    print(f"Train date range: {train_dates[0]} to {train_dates[-1]}")
    print(f"Valid date range: {valid_dates[0]} to {valid_dates[-1]}")
else:
    train_df_filtered = train_df

In [ ]:
# Custom weighted MAE metric for multi-target
def weighted_mae(y_true, y_pred, weights_dict=TARGET_WEIGHTS):
    """
    Calculate weighted MAE across multiple targets
    y_true, y_pred: shape (n_samples, 3) for [short, medium, long]
    """
    mae_short = np.mean(np.abs(y_pred[:, 0] - y_true[:, 0]))
    mae_medium = np.mean(np.abs(y_pred[:, 1] - y_true[:, 1]))
    mae_long = np.mean(np.abs(y_pred[:, 2] - y_true[:, 2]))
    
    weighted_score = (mae_short * weights_dict['short'] + 
                      mae_medium * weights_dict['medium'] + 
                      mae_long * weights_dict['long'])
    return weighted_score

# Custom metric for LightGBM (for single target)
def mae_lgb(y_true, y_pred):
    mae = np.mean(np.abs(y_pred - y_true))
    return 'mae', mae, False

# Custom metric for XGBoost (for single target)
def mae_xgb(y_true, y_pred):
    mae = np.mean(np.abs(y_pred - y_true))
    return 'mae', mae

print("Metrics defined successfully.")

In [ ]:
# Prepare validation data
if TRAINING:
    X_valid = train_df_filtered[feature_cols].loc[train_df_filtered['date_id'].isin(valid_dates)].values
    y_valid = train_df_filtered[target_cols].loc[train_df_filtered['date_id'].isin(valid_dates)].values
    
    print(f"Validation set shape: X={X_valid.shape}, y={y_valid.shape}")
    print(f"Validation set - Short target range: [{y_valid[:, 0].min():.4f}, {y_valid[:, 0].max():.4f}]")
    print(f"Validation set - Medium target range: [{y_valid[:, 1].min():.4f}, {y_valid[:, 1].max():.4f}]")
    print(f"Validation set - Long target range: [{y_valid[:, 2].min():.4f}, {y_valid[:, 2].max():.4f}]")


#draw graph of target distribution
plt.figure(figsize=(10, 6))
plt.boxplot([train_df_filtered['target_short'], train_df_filtered['target_medium'], train_df_filtered['target_long']],
            labels=['Target Short', 'Target Medium', 'Target Long'])
plt.title('Target Value Distribution')
plt.ylabel('Target Values')
plt.show()


In [ ]:
# Training function - Multi-output version (one model predicts all 3 targets)
def train_model(fold_idx, model_name='lgb'):
    """
    Train a multi-output model for all targets at once
    
    Args:
        fold_idx: Fold number (0 to N_fold-1)
        model_name: Model type ('lgb', 'xgb', 'cbt')
    """
    print(f"\n{'='*80}")
    print(f"Training {model_name.upper()} - Fold {fold_idx+1}/{N_fold} - Multi-output (3 targets)")
    print(f"{'='*80}")
    
    if TRAINING:
        # Select dates for this fold (leave out every N_fold-th date)
        selected_dates = [date for ii, date in enumerate(train_dates) if ii % N_fold != fold_idx]
        print(f"Training on {len(selected_dates)} dates")
        
        # Prepare training data - ALL 3 TARGETS
        train_mask = train_df_filtered['date_id'].isin(selected_dates)
        X_train = train_df_filtered[feature_cols].loc[train_mask].values
        y_train = train_df_filtered[target_cols].loc[train_mask].values  # Shape: (n, 3)
        
        print(f"Train: X={X_train.shape}, y={y_train.shape}")
        print(f"Valid: X={X_valid.shape}, y={y_valid.shape}")
        
        # Initialize model based on type
        if model_name == 'lgb':
            # LightGBM doesn't support multi-output directly, need to use MultiOutputRegressor
            from sklearn.multioutput import MultiOutputRegressor
            
            base_model = lgb.LGBMRegressor(
                n_estimators=1000,
                learning_rate=0.05,
                max_depth=8,
                num_leaves=64,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42 + fold_idx,
                device='cpu',  # 使用CPU（GPU未编译）
                verbose=-1
            )
            
            model = MultiOutputRegressor(base_model)
            model.fit(X_train, y_train)
            
        elif model_name == 'xgb':
            # XGBoost supports multi-output with MultiOutputRegressor
            from sklearn.multioutput import MultiOutputRegressor
            
            base_model = xgb.XGBRegressor(
                n_estimators=1000,
                learning_rate=0.05,
                max_depth=8,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42 + fold_idx,
                tree_method='hist',
                device='gpu',
                eval_metric='mae'
            )
            
            model = MultiOutputRegressor(base_model)
            model.fit(X_train, y_train)
            
        elif model_name == 'cbt':
            # CatBoost supports multi-output natively with loss_function='MultiRMSE'
            model = cbt.CatBoostRegressor(
                iterations=1000,
                learning_rate=0.05,
                depth=8,
                random_state=42 + fold_idx,
                task_type='GPU',
                loss_function='MultiRMSE',  # Multi-output loss
                verbose=50
            )
            
            model.fit(
                X_train, y_train,
                eval_set=[(X_valid, y_valid)],
                early_stopping_rounds=100,
                verbose=50
            )
        
        # Save model
        model_filename = f'./models/{model_name}_fold{fold_idx}.model'
        joblib.dump(model, model_filename)
        print(f"Model saved to {model_filename}")
        
        # Evaluate on validation set - all targets
        y_pred = model.predict(X_valid)
        
        print(f"\nValidation MAE by target:")
        for idx, target_name in enumerate(['short', 'medium', 'long']):
            mae = np.mean(np.abs(y_pred[:, idx] - y_valid[:, idx]))
            print(f"  {target_name.upper():8s}: {mae:.6f}")
        
        # Calculate weighted MAE
        val_weighted_mae = weighted_mae(y_valid, y_pred)
        print(f"  Weighted MAE: {val_weighted_mae:.6f}")
        
        # Clean up memory
        del X_train, y_train
        import gc
        gc.collect()
        
        return model, val_weighted_mae
    else:
        # Load pre-trained model
        model_filename = f'./models/{model_name}_fold{fold_idx}.model'
        model = joblib.load(model_filename)
        print(f"Model loaded from {model_filename}")
        return model, None

print("Training function defined (multi-output version).")

In [ ]:
# Dictionary to store all trained models
# Structure: models[model_name][fold_idx] = model (each model predicts 3 targets)
models = {
    'lgb': [],
    'xgb': [],
    'cbt': []
}

# Dictionary to store validation scores (weighted MAE)
val_scores = {
    'lgb': [],
    'xgb': [],
    'cbt': []
}

print("Model storage initialized (multi-output version).")

## 1.1 Train LightGBM Models

In [ ]:
# Train LightGBM models for all folds
if TRAINING:
    print("="*80)
    print("TRAINING LIGHTGBM MODELS (Multi-output)")
    print("="*80)
    
    for fold_idx in range(N_fold):
        model, val_mae = train_model(fold_idx, 'lgb')
        models['lgb'].append(model)
        val_scores['lgb'].append(val_mae)
    
    # Print summary
    print("\n" + "="*80)
    print("LIGHTGBM VALIDATION SUMMARY")
    print("="*80)
    avg_mae = np.mean(val_scores['lgb'])
    std_mae = np.std(val_scores['lgb'])
    print(f"Weighted MAE across {N_fold} folds: {avg_mae:.6f} ± {std_mae:.6f}")
    print(f"Best fold: {np.min(val_scores['lgb']):.6f}")
    print(f"Worst fold: {np.max(val_scores['lgb']):.6f}")
else:
    print("Skipping LightGBM training (TRAINING=False)")

## 1.2 Train XGBoost Models

In [ ]:
# Train XGBoost models for all folds
if TRAINING:
    print("="*80)
    print("TRAINING XGBOOST MODELS (Multi-output)")
    print("="*80)
    
    for fold_idx in range(N_fold):
        model, val_mae = train_model(fold_idx, 'xgb')
        models['xgb'].append(model)
        val_scores['xgb'].append(val_mae)
    
    # Print summary
    print("\n" + "="*80)
    print("XGBOOST VALIDATION SUMMARY")
    print("="*80)
    avg_mae = np.mean(val_scores['xgb'])
    std_mae = np.std(val_scores['xgb'])
    print(f"Weighted MAE across {N_fold} folds: {avg_mae:.6f} ± {std_mae:.6f}")
    print(f"Best fold: {np.min(val_scores['xgb']):.6f}")
    print(f"Worst fold: {np.max(val_scores['xgb']):.6f}")
else:
    print("Skipping XGBoost training (TRAINING=False)")

## 1.3 Train CatBoost Models

In [ ]:
# Train CatBoost models for all folds
if TRAINING:
    print("="*80)
    print("TRAINING CATBOOST MODELS (Multi-output)")
    print("="*80)
    
    for fold_idx in range(N_fold):
        model, val_mae = train_model(fold_idx, 'cbt')
        models['cbt'].append(model)
        val_scores['cbt'].append(val_mae)
    
    # Print summary
    print("\n" + "="*80)
    print("CATBOOST VALIDATION SUMMARY")
    print("="*80)
    avg_mae = np.mean(val_scores['cbt'])
    std_mae = np.std(val_scores['cbt'])
    print(f"Weighted MAE across {N_fold} folds: {avg_mae:.6f} ± {std_mae:.6f}")
    print(f"Best fold: {np.min(val_scores['cbt']):.6f}")
    print(f"Worst fold: {np.max(val_scores['cbt']):.6f}")
else:
    print("Skipping CatBoost training (TRAINING=False)")

# 2. Model Evaluation and Ensemble

In [ ]:
# Compare all models
if TRAINING:
    print("\n" + "="*80)
    print("OVERALL MODEL COMPARISON")
    print("="*80)
    
    results_summary = []
    
    for model_name in ['lgb', 'xgb', 'cbt']:
        scores = val_scores[model_name]
        results_summary.append({
            'Model': model_name.upper(),
            'Mean Weighted MAE': np.mean(scores),
            'Std': np.std(scores),
            'Min': np.min(scores),
            'Max': np.max(scores)
        })
    
    results_df = pd.DataFrame(results_summary)
    print(results_df.to_string(index=False))
    
    # Find best model
    best_model = min(['lgb', 'xgb', 'cbt'], key=lambda x: np.mean(val_scores[x]))
    print(f"\n最佳模型: {best_model.upper()} (Weighted MAE: {np.mean(val_scores[best_model]):.6f})")
else:
    print("Skipping model comparison (TRAINING=False)")

## 2.1 IC分析 - 评估CatBoost的排序能力

In [ ]:
from scipy.stats import spearmanr, pearsonr

def calculate_ic_metrics(y_true, y_pred, dates, target_names=['short', 'medium', 'long']):
    """
    计算IC (Information Coefficient) 指标
    
    Args:
        y_true: 真实值 (n_samples, 3)
        y_pred: 预测值 (n_samples, 3)
        dates: 日期ID (n_samples,)
        target_names: target名称列表
    
    Returns:
        ic_results: IC分析结果字典
    """
    results = {}
    
    for idx, target_name in enumerate(target_names):
        # 提取单个target的真实值和预测值
        y_true_target = y_true[:, idx]
        y_pred_target = y_pred[:, idx]
        
        # 1. 总体IC (Pearson和Spearman)
        ic_pearson, _ = pearsonr(y_true_target, y_pred_target)
        ic_spearman, _ = spearmanr(y_true_target, y_pred_target)
        
        # 2. 按日期计算IC (逐日IC)
        unique_dates = np.unique(dates)
        daily_ic_pearson = []
        daily_ic_spearman = []
        
        for date in unique_dates:
            mask = dates == date
            if np.sum(mask) > 1:  # 确保有足够样本
                y_true_day = y_true_target[mask]
                y_pred_day = y_pred_target[mask]
                
                # 计算当日IC
                ic_p, _ = pearsonr(y_true_day, y_pred_day)
                ic_s, _ = spearmanr(y_true_day, y_pred_day)
                
                daily_ic_pearson.append(ic_p)
                daily_ic_spearman.append(ic_s)
        
        daily_ic_pearson = np.array(daily_ic_pearson)
        daily_ic_spearman = np.array(daily_ic_spearman)
        
        # 3. 统计IC指标
        results[target_name] = {
            'ic_pearson': ic_pearson,
            'ic_spearman': ic_spearman,
            'daily_ic_mean': np.mean(daily_ic_pearson),
            'daily_ic_std': np.std(daily_ic_pearson),
            'daily_ic_spearman_mean': np.mean(daily_ic_spearman),
            'daily_ic_positive_rate': np.mean(daily_ic_pearson > 0),  # IC>0的比例
            'daily_ic_positive_rate_spearman': np.mean(daily_ic_spearman > 0),
            'daily_ic_values': daily_ic_pearson,
            'daily_ic_spearman_values': daily_ic_spearman
        }
    
    return results

print("IC分析函数定义完成。")

In [ ]:
# 计算CatBoost在验证集上的IC
if TRAINING:
    print("="*80)
    print("CATBOOST IC分析 - 验证集")
    print("="*80)
    
    # 获取CatBoost的预测（所有fold平均）
    cbt_fold_preds = []
    for model in models['cbt']:
        cbt_fold_preds.append(model.predict(X_valid))
    cbt_preds = np.mean(cbt_fold_preds, axis=0)
    
    # 获取验证集的日期
    valid_dates_array = train_df_filtered['date_id'].loc[
        train_df_filtered['date_id'].isin(valid_dates)
    ].values
    
    # 计算IC
    ic_results = calculate_ic_metrics(y_valid, cbt_preds, valid_dates_array)
    
    # 打印结果
    print("\n1. 总体IC指标:")
    print("-" * 80)
    for target_name in ['short', 'medium', 'long']:
        res = ic_results[target_name]
        print(f"\n{target_name.upper()}:")
        print(f"  Overall Pearson IC:  {res['ic_pearson']:.4f}")
        print(f"  Overall Spearman IC: {res['ic_spearman']:.4f}")
    
    print("\n2. 逐日IC统计:")
    print("-" * 80)
    ic_summary = []
    for target_name in ['short', 'medium', 'long']:
        res = ic_results[target_name]
        ic_summary.append({
            'Target': target_name.upper(),
            'Mean IC': res['daily_ic_mean'],
            'IC Std': res['daily_ic_std'],
            'Spearman Mean IC': res['daily_ic_spearman_mean'],
            'IC>0 Rate (%)': res['daily_ic_positive_rate'] * 100,
            'Spearman IC>0 (%)': res['daily_ic_positive_rate_spearman'] * 100
        })
    
    ic_df = pd.DataFrame(ic_summary)
    print(ic_df.to_string(index=False))
    
    # 3. IC解读
    print("\n3. IC指标解读:")
    print("-" * 80)
    print("IC (Information Coefficient) 衡量预测值和真实值的相关性：")
    print("  • |IC| > 0.05: 具有一定预测能力")
    print("  • |IC| > 0.10: 预测能力较强")
    print("  • |IC| > 0.15: 预测能力很强")
    print("  • IC>0率: 衡量模型稳定性（理想 > 55%）")
    
    print("\n4. CatBoost表现评估:")
    print("-" * 80)
    for target_name in ['short', 'medium', 'long']:
        res = ic_results[target_name]
        ic_val = abs(res['ic_pearson'])
        ic_pos_rate = res['daily_ic_positive_rate'] * 100
        
        # 评级
        if ic_val > 0.15:
            rating = "🌟🌟🌟 优秀"
        elif ic_val > 0.10:
            rating = "🌟🌟 良好"
        elif ic_val > 0.05:
            rating = "🌟 一般"
        else:
            rating = "⚠️  较弱"
        
        stability = "稳定" if ic_pos_rate > 55 else "不稳定"
        
        print(f"\n{target_name.upper()}: {rating}")
        print(f"  - Pearson IC = {res['ic_pearson']:.4f}")
        print(f"  - IC>0率 = {ic_pos_rate:.1f}% ({stability})")
else:
    print("跳过IC分析 (TRAINING=False)")

In [ ]:
# 可视化IC的时间序列
if TRAINING:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    for idx, target_name in enumerate(['short', 'medium', 'long']):
        ax = axes[idx]
        
        # 获取逐日IC值
        daily_ic = ic_results[target_name]['daily_ic_values']
        daily_ic_spearman = ic_results[target_name]['daily_ic_spearman_values']
        
        # 绘制IC时间序列
        x = range(len(daily_ic))
        ax.plot(x, daily_ic, label='Pearson IC', alpha=0.7, linewidth=1)
        ax.plot(x, daily_ic_spearman, label='Spearman IC', alpha=0.7, linewidth=1)
        
        # 添加均值线
        ax.axhline(y=np.mean(daily_ic), color='red', linestyle='--', 
                   label=f'Mean IC={np.mean(daily_ic):.4f}', linewidth=1.5)
        ax.axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=0.8)
        
        # 添加±1std区域
        mean_ic = np.mean(daily_ic)
        std_ic = np.std(daily_ic)
        ax.fill_between(x, mean_ic - std_ic, mean_ic + std_ic, 
                        alpha=0.2, color='gray', label='±1 Std')
        
        ax.set_title(f'Target {target_name.upper()} - Daily IC Time Series', fontsize=12, fontweight='bold')
        ax.set_xlabel('Validation Date Index')
        ax.set_ylabel('IC Value')
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # IC分布直方图
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for idx, target_name in enumerate(['short', 'medium', 'long']):
        ax = axes[idx]
        daily_ic = ic_results[target_name]['daily_ic_values']
        
        ax.hist(daily_ic, bins=30, alpha=0.7, edgecolor='black')
        ax.axvline(x=np.mean(daily_ic), color='red', linestyle='--', 
                  label=f'Mean={np.mean(daily_ic):.4f}', linewidth=2)
        ax.axvline(x=0, color='black', linestyle='-', alpha=0.5, linewidth=1)
        
        ax.set_title(f'{target_name.upper()} - IC Distribution')
        ax.set_xlabel('IC Value')
        ax.set_ylabel('Frequency')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ IC可视化完成")

In [ ]:
# 排序能力分析 - Quintile Analysis (五分位数分析)
if TRAINING:
    print("\n" + "="*80)
    print("排序能力分析 - 五分位数收益")
    print("="*80)
    print("将预测值分成5组，看真实收益的分布\n")
    
    for target_name_idx, target_name in enumerate(['short', 'medium', 'long']):
        print(f"\n{target_name.upper()}:")
        print("-" * 60)
        
        # 获取预测和真实值
        y_pred_target = cbt_preds[:, target_name_idx]
        y_true_target = y_valid[:, target_name_idx]
        
        # 按预测值分成5组（quintiles）
        quintiles = pd.qcut(y_pred_target, q=5, labels=['Q1(Low)', 'Q2', 'Q3', 'Q4', 'Q5(High)'], duplicates='drop')
        
        # 计算每组的平均真实收益
        quintile_analysis = pd.DataFrame({
            'Predicted_Quintile': quintiles,
            'True_Return': y_true_target
        })
        
        quintile_stats = quintile_analysis.groupby('Predicted_Quintile')['True_Return'].agg([
            ('Count', 'count'),
            ('Mean_Return', 'mean'),
            ('Std_Return', 'std'),
            ('Median_Return', 'median')
        ]).reset_index()
        
        print(quintile_stats.to_string(index=False))
        
        # 计算多空收益（Q5 - Q1）
        if len(quintile_stats) >= 2:
            long_return = quintile_stats.iloc[-1]['Mean_Return']  # 最高分位
            short_return = quintile_stats.iloc[0]['Mean_Return']   # 最低分位
            long_short = long_return - short_return
            
            print(f"\n多空收益 (Q5-Q1): {long_short:.6f}")
            
            if long_short > 0:
                print("✓ 正向排序能力：预测高的确实收益更高")
            else:
                print("✗ 排序能力较弱：预测和真实收益不匹配")
    
    print("\n" + "="*80)
    print("解读说明:")
    print("  • 理想情况：Q5 > Q4 > Q3 > Q2 > Q1 (单调递增)")
    print("  • 多空收益 > 0 且显著：说明模型有良好的排序能力")
    print("  • 结合IC>0率：两者都好说明模型稳定且有效")
    print("="*80)

In [ ]:
# Function to make ensemble predictions (multi-output version)
def ensemble_predict(X, models_dict, model_types=['lgb', 'xgb', 'cbt'], ensemble_weights=None):
    """
    Make ensemble predictions across all models
    
    Args:
        X: Input features
        models_dict: Dictionary of models
        model_types: List of model types to use
        ensemble_weights: Optional weights for each model type (defaults to equal weighting)
    
    Returns:
        predictions: Array of shape (n_samples, 3) for [short, medium, long]
    """
    if ensemble_weights is None:
        ensemble_weights = {m: 1.0/len(model_types) for m in model_types}
    
    predictions = np.zeros((len(X), 3))
    
    for model_type in model_types:
        # Average predictions across all folds for this model type
        fold_preds = []
        for model in models_dict[model_type]:
            fold_preds.append(model.predict(X))
        
        # Average across folds
        avg_fold_pred = np.mean(fold_preds, axis=0)  # Shape: (n_samples, 3)
        
        # Add weighted prediction
        predictions += avg_fold_pred * ensemble_weights[model_type]
    
    return predictions

print("Ensemble prediction function defined (multi-output version).")

In [ ]:
# Validate ensemble on validation set
if TRAINING:
    print("\n" + "="*80)
    print("ENSEMBLE VALIDATION")
    print("="*80)
    
    # Equal weighting ensemble
    ensemble_preds_equal = ensemble_predict(X_valid, models)
    weighted_mae_ensemble = weighted_mae(y_valid, ensemble_preds_equal)
    
    print(f"\nEqual-weighted ensemble (1/3, 1/3, 1/3):")
    for idx, target_name in enumerate(['short', 'medium', 'long']):
        mae = np.mean(np.abs(ensemble_preds_equal[:, idx] - y_valid[:, idx]))
        print(f"  {target_name.upper():8s} MAE: {mae:.6f}")
    print(f"  Weighted MAE: {weighted_mae_ensemble:.6f}")
    
    # Performance-weighted ensemble (inverse of validation MAE)
    print(f"\nCalculating performance-weighted ensemble...")
    
    # Calculate weights based on inverse of validation MAE
    model_weights = {}
    for model_type in ['lgb', 'xgb', 'cbt']:
        avg_score = np.mean(val_scores[model_type])
        model_weights[model_type] = 1.0 / avg_score
    
    # Normalize weights
    total_weight = sum(model_weights.values())
    model_weights = {k: v/total_weight for k, v in model_weights.items()}
    
    print(f"Performance-based weights: {model_weights}")
    
    ensemble_preds_weighted = ensemble_predict(X_valid, models, ensemble_weights=model_weights)
    weighted_mae_ensemble_perf = weighted_mae(y_valid, ensemble_preds_weighted)
    
    print(f"\nPerformance-weighted ensemble:")
    for idx, target_name in enumerate(['short', 'medium', 'long']):
        mae = np.mean(np.abs(ensemble_preds_weighted[:, idx] - y_valid[:, idx]))
        print(f"  {target_name.upper():8s} MAE: {mae:.6f}")
    print(f"  Weighted MAE: {weighted_mae_ensemble_perf:.6f}")
    
    # Comparison
    print(f"\n" + "="*80)
    print("ENSEMBLE方法对比:")
    print(f"  Equal-weighted:       {weighted_mae_ensemble:.6f}")
    print(f"  Performance-weighted: {weighted_mae_ensemble_perf:.6f}")
    if weighted_mae_ensemble_perf < weighted_mae_ensemble:
        print(f"  ✓ Performance-weighted更好 (改善 {(weighted_mae_ensemble - weighted_mae_ensemble_perf):.6f})")
    else:
        print(f"  ✓ Equal-weighted更好")
else:
    print("Skipping ensemble validation (TRAINING=False)")

# 3. Generate Test Set Predictions

In [ ]:
# Generate predictions on test set
print("="*80)
print("GENERATING TEST PREDICTIONS")
print("="*80)

# Use the better performing ensemble method
if TRAINING and weighted_mae_ensemble_perf < weighted_mae_ensemble:
    print("Using performance-weighted ensemble")
    test_predictions = ensemble_predict(X_test, models, ensemble_weights=model_weights)
else:
    print("Using equal-weighted ensemble")
    test_predictions = ensemble_predict(X_test, models)

print(f"\nTest predictions shape: {test_predictions.shape}")
print(f"Prediction ranges:")
for idx, target_name in enumerate(['short', 'medium', 'long']):
    print(f"  {target_name.upper():8s}: [{test_predictions[:, idx].min():.4f}, {test_predictions[:, idx].max():.4f}]")

# Create submission DataFrame
submission = pd.DataFrame({
    'id': test_df.index,
    'target_short': test_predictions[:, 0],
    'target_medium': test_predictions[:, 1],
    'target_long': test_predictions[:, 2]
})

print(f"\nSubmission shape: {submission.shape}")
print(submission.head(10))

In [ ]:
# Save submission file
submission_filename = 'submission_ensemble.csv'
submission.to_csv(submission_filename, index=False)
print(f"\n✓ Submission saved to: {submission_filename}")

# Also save individual model predictions for analysis
if TRAINING:
    for model_type in ['lgb', 'xgb', 'cbt']:
        model_preds = ensemble_predict(X_test, models, model_types=[model_type])
        
        submission_single = pd.DataFrame({
            'row_id': test_df.index,
            'target_short': model_preds[:, 0],
            'target_medium': model_preds[:, 1],
            'target_long': model_preds[:, 2]
        })
        
        filename = f'submission_{model_type}.csv'
        submission_single.to_csv(filename, index=False)
        print(f"✓ {model_type.upper()} submission saved to: {filename}")